### CEO Knowledge Agents

Creates Agent Bricks Knowledge Assistants for the 4 CEO-specific document corpora:
- **Legal Complaints KA** — employment, liability, vendor disputes
- **Regulatory Documents KA** — permits, fire safety, zoning, FDA
- **Audit Reports KA** — financial, operational, food safety, supply chain audits
- **Consultancy Reports KA** — strategy, ops efficiency, AI transformation, workforce

In [ ]:
CATALOG = dbutils.widgets.get("CATALOG")
LEGAL_ENDPOINT   = dbutils.widgets.get("CEO_LEGAL_KA_ENDPOINT")
REG_ENDPOINT     = dbutils.widgets.get("CEO_REGULATORY_KA_ENDPOINT")
AUDIT_ENDPOINT   = dbutils.widgets.get("CEO_AUDIT_KA_ENDPOINT")
CONSULT_ENDPOINT = dbutils.widgets.get("CEO_CONSULTANCY_KA_ENDPOINT")

In [ ]:
from databricks.sdk import WorkspaceClient
import json, sys
from concurrent.futures import ThreadPoolExecutor, as_completed

sys.path.append('../utils')
from uc_state import add

w = WorkspaceClient()
API_BASE = "/api/2.0/knowledge-assistants"

KA_CONFIGS = [
    {
        "name": f"{CATALOG}-ceo-legal",
        "endpoint": LEGAL_ENDPOINT,
        "volume_path": f"/Volumes/{CATALOG}/legal_complaints/documents",
        "source_name": "legal_complaint_documents",
        "description": (
            "Answers CEO-level questions about Casper's Kitchens legal exposure. "
            "Covers employment disputes (discrimination, wrongful termination, wage violations), "
            "customer liability claims, and vendor/contract disputes across all 8 locations (US + EMEA)."
        ),
        "source_description": (
            "Legal complaint PDFs for Casper's Kitchens Inc. Each document is a confidential "
            "attorney-client privileged case file covering case number, parties, legal basis, "
            "applicable statutes, relief sought, case status, and risk assessment."
        ),
        "instructions": (
            "You are a legal intelligence assistant for the CEO of Casper's Kitchens. "
            "Always cite the specific case number and location when answering. "
            "Clearly state the risk level (HIGH/MEDIUM/LOW) and amount at stake. "
            "Remind the CEO that all legal decisions should be made with counsel. "
            "Never speculate on legal outcomes."
        ),
    },
    {
        "name": f"{CATALOG}-ceo-regulatory",
        "endpoint": REG_ENDPOINT,
        "volume_path": f"/Volumes/{CATALOG}/regulatory/documents",
        "source_name": "regulatory_documents",
        "description": (
            "Answers questions about Casper's Kitchens regulatory compliance status. "
            "Covers food service permits, fire safety certificates, zoning compliance, "
            "FDA registrations, and food handler certifications across all 8 locations (US + EMEA)."
        ),
        "source_description": (
            "Regulatory compliance PDFs for all 4 Casper's Kitchens ghost kitchen locations. "
            "Includes document type, issuing authority, issue date, expiry date, current status, "
            "and specific conditions and requirements from each regulatory body."
        ),
        "instructions": (
            "You are a regulatory compliance assistant for the CEO. "
            "Always cite the specific document ID, issuing authority, and expiry date. "
            "Flag any permits or certificates that are expiring within 60 days or have a conditional status. "
            "Summarize compliance risk clearly and concisely."
        ),
    },
    {
        "name": f"{CATALOG}-ceo-audits",
        "endpoint": AUDIT_ENDPOINT,
        "volume_path": f"/Volumes/{CATALOG}/audits/reports",
        "source_name": "audit_reports",
        "description": (
            "Answers CEO-level questions about Casper's Kitchens audit findings. "
            "Covers financial statement audits, operational compliance, food safety management, "
            "and supply chain audits across all locations and quarters."
        ),
        "source_description": (
            "Independent audit reports prepared by Big 4 and mid-tier firms for Casper's Kitchens. "
            "Each report includes audit type, period, scope, auditor's opinion, individual findings "
            "(Critical/Significant/Minor/Informational) with remediation details."
        ),
        "instructions": (
            "You are an audit intelligence assistant for the CEO. "
            "Always cite the specific audit ID, auditing firm, and period covered. "
            "Highlight Critical and Significant findings prominently. "
            "Summarize the auditor's opinion and any patterns across multiple audits. "
            "Be specific about financial impacts and remediation deadlines."
        ),
    },
    {
        "name": f"{CATALOG}-ceo-consultancy",
        "endpoint": CONSULT_ENDPOINT,
        "volume_path": f"/Volumes/{CATALOG}/consultancy/reports",
        "source_name": "consultancy_reports",
        "description": (
            "Answers questions about strategic recommendations from management consultants. "
            "Covers market expansion, operations efficiency, AI transformation, and workforce "
            "management reports from top-tier consulting firms."
        ),
        "source_description": (
            "Management consulting reports prepared exclusively for Casper's Kitchens. "
            "Each report includes executive summary, detailed section-by-section analysis, "
            "financial projections, and specific recommendations with ROI estimates."
        ),
        "instructions": (
            "You are a strategic intelligence assistant for the CEO. "
            "Always cite the consulting firm and report date. "
            "Summarize key recommendations with financial impact figures. "
            "When multiple reports address the same topic, synthesize the consensus view. "
            "Be direct and executive-ready in your summaries."
        ),
    },
]


def _id_from_post(resp):
    if isinstance(resp, dict):
        ka = resp.get("knowledge_assistant", resp)
        return resp.get("tile_id") or ka.get("tile_id") or ka.get("id")
    return None


# Step 1: Check uc_state first — fast Spark SQL, zero API calls
known_kas = {}
try:
    df = spark.sql(f"""
        SELECT resource_data FROM {CATALOG}._internal_state.resources
        WHERE resource_type = 'knowledge_assistants'
        ORDER BY created_at DESC
    """)
    for row in df.collect():
        info = json.loads(row.resource_data)
        ep = info.get("endpoint_name", "")
        kid = info.get("tile_id", "")
        if ep and kid and ep not in known_kas:
            known_kas[ep] = kid
    print(f"uc_state: found {len(known_kas)} KAs")
except Exception as e:
    print(f"⚠️ uc_state lookup failed: {e}")

# Step 2: Only call the KA API for KAs not already in uc_state
missing = [cfg for cfg in KA_CONFIGS if cfg["endpoint"] not in known_kas]
if missing:
    print(f"{len(missing)} KAs missing from uc_state — checking KA API...")
    try:
        params = {}
        while True:
            resp = w.api_client.do("GET", API_BASE, query=params)
            for item in resp.get("knowledge_assistants", []):
                ka = item.get("knowledge_assistant", item)
                ep = ka.get("endpoint_name", "")
                kid = item.get("tile_id") or ka.get("tile_id") or ka.get("id")
                if ep and kid and ep not in known_kas:
                    known_kas[ep] = kid
            token = resp.get("next_page_token")
            if not token:
                break
            params = {"page_token": token}
    except Exception as e:
        print(f"⚠️ KA list error: {e}")


def _create_and_wait(cfg):
    name = cfg["name"]

    if cfg["endpoint"] in known_kas:
        agent_id = known_kas[cfg["endpoint"]]
        print(f"[{name}] ♻️  Already exists ({agent_id}) — skipping")
        return {"name": name, "tile_id": agent_id, "endpoint": cfg["endpoint"]}

    print(f"[{name}] Creating...")
    ka_body = {
        "name": name,
        "description": cfg["description"],
        "endpoint_name": cfg["endpoint"],
        "knowledge_sources": [{"files_source": {
            "name": cfg["source_name"], "type": "files",
            "files": {"path": cfg["volume_path"]},
            "description": cfg["source_description"],
        }}],
        "instructions": cfg["instructions"],
    }
    resp = w.api_client.do("POST", API_BASE, body=ka_body)
    agent_id = _id_from_post(resp)
    print(f"[{name}] ✅ Created: {agent_id} — readiness polling handled by CEO_Readiness_Check")

    return {"name": name, "tile_id": agent_id, "endpoint": cfg["endpoint"]}


# Run in parallel; collect results then write to uc_state on the main thread
created_agents = []
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(_create_and_wait, cfg): cfg for cfg in KA_CONFIGS}
    for future in as_completed(futures):
        result = future.result()
        created_agents.append(result)

for r in created_agents:
    add(CATALOG, "knowledge_assistants", {"endpoint_name": r["endpoint"], "tile_id": r["tile_id"], "name": r["name"]})
    print(f"   Registered: {r['name']} ({r['tile_id']})")

print(f"\n✅ CEO Knowledge Agents stage complete — {len(created_agents)} KAs processed")
